In [5]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:   ", "✅" if GROQ_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:    ✅


In [6]:
from pageindex import PageIndexClient
from groq import Groq

# Initialize PageIndex
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# Initialize Groq
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


### Section 2: Upload & Index a PDF

#### What happens here:

1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

#### Why NO chunking?

Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [7]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "C:\Agentic AI\Agentic_AI_Course\Vectorless RAG\Latest Trends in Biomedical Instrumentation.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: C:\Agentic AI\Agentic_AI_Course\Vectorless RAG\Latest Trends in Biomedical Instrumentation.pdf
✅ Uploaded!
📋 Document ID: pi-cmq6yz09l005a01quzzuxdta8
   (Save this ID — you'll use it throughout the notebook)


In [8]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


### Section 3: Inspect the Tree Structure

#### What the tree looks like:

```text
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

#### Each node has:

* `node_id` — unique ID used during retrieval
* `title` — section heading
* `page_index` — page number in original PDF
* `text` — section summary (when `node_summary=True`)
* `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [9]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 11

🌲 Raw tree (first node):
{
  "title": "Latest Trends in Biomedical Instrumentation",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# Latest Trends in Biomedical Instrumentation\n",
  "text": "# Latest Trends in Biomedical Instrumentation\n"
}


In [10]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Latest Trends in Biomedical Instrumentation  (p.1)
[0001] Introduction  (p.2)
  └─ [0002] Need for Advanced Instrumentation  (p.3)
  └─ [0003] Classification of Biomedical Instruments  (p.4)
  └─ [0004] Trend 1 – Wearable Health Devices  (p.5)
[0005] Wearable Device Working  (p.6)
  └─ [0006] Trend 2 – Internet of Medical Things (IoMT)  (p.7)
[0007] IoMT Architecture  (p.8)
[0008] AI in ECG Analysis  (p.10)
[0009] Trend 4 – Telemedicine  (p.11)
  └─ [0010] Trend 5 – Point of Care (POC) Devices  (p.12)
[0011] Trend 6 – Implantable Devices  (p.13)
[0012] Trend 7 – Robotics in Surgery  (p.14)
[0013] Trend 9 – Lab-on-Chip Technology  (p.16)
[0014] Challenges  (p.17)
[0015] Future Scope  (p.18)


In [11]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 16
   Each node = one retrievable section of the document


### Section 4: LLM Tree Search — The Core of PageIndex

This is where PageIndex fundamentally differs from vector RAG.

#### Vector RAG retrieval:

```text
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```

**Problem:** finds what's similar, not what's relevant


#### PageIndex retrieval:

```text
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```

**Advantage:** LLM understands document structure, context, and intent

**The LLM acts like a human expert scanning a Table of Contents.**


In [19]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "llama-3.3-70b-versatile") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [25]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What are the latest trends in Biomedical enginnering?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What are the latest trends in Biomedical enginnering?

🧠 LLM Reasoning:
To identify the latest trends in Biomedical engineering, we need to look for sections in the document tree that directly mention trends or advancements in the field. Starting from the top, node '0000' is titled 'Latest Trends in Biomedical Instrumentation,' which directly relates to the query. However, to get a comprehensive understanding, we should also consider child nodes of major sections that discuss trends. Node '0001' is 'Introduction,' which might set the stage but doesn't directly answer the question. Its child nodes, such as '0004' (Trend 1 – Wearable Health Devices), are more relevant. Similarly, other nodes like '0006' (Trend 2 – Internet of Medical Things (IoMT)), '0007' (IoMT Architecture), '0008' (AI in ECG Analysis), '0009' (Trend 4 – Telemedicine), '0010' (Trend 5 – Point of Care (POC) Devices), '0011' (Trend 6 – Implantable Devices), '0012' (Trend 7 – Robotics in Surgery), '0013' (Trend 9

### Section 5: Full End-to-End RAG Pipeline

#### 3 Steps:

1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes
3. **Generate** → LLM writes a grounded answer with page citations

#### What makes this better than vector RAG:

* Retrieved content has titles + page numbers (traceable)
* LLM can cite exactly *which section* the answer comes from
* No hallucination from irrelevant chunks


In [26]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [33]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "llama-3.3-70b-versatile") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [34]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [35]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the latest trends in Biomedical enginnering?",
    tree=pageindex_tree
)

🔍 Query: What are the latest trends in Biomedical enginnering?

🧠 Reasoning: To identify the node IDs that most likely contain the answer to the query 'What are the latest trends in Biomedical engineering?', we start by looking for titles and summaries that directly mention 't...
🎯 Retrieved node IDs: ['0000', '0004', '0006', '0009', '0010', '0011', '0012', '0013']
📄 Sections found: ['Latest Trends in Biomedical Instrumentation', 'Trend 1 – Wearable Health Devices', 'Trend 2 – Internet of Medical Things (IoMT)', 'Trend 4 – Telemedicine', 'Trend 5 – Point of Care (POC) Devices', 'Trend 6 – Implantable Devices', 'Trend 7 – Robotics in Surgery', 'Trend 9 – Lab-on-Chip Technology']

📝 Answer:
The latest trends in Biomedical engineering include Wearable Health Devices (Trend 1 – Wearable Health Devices | Page 5), Internet of Medical Things (IoMT) (Trend 2 – Internet of Medical Things (IoMT) | Page 7), Telemedicine (Trend 4 – Telemedicine | Page 11), Point of Care (POC) Devices (Trend 5 – Po

In [36]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What are the latest trends in Biomedical enginnering?",
    "What are the future scopes of biomedical engineering?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the latest trends in Biomedical enginnering?
A: The latest trends in Biomedical engineering include Wearable Health Devices (Trend 1 – Wearable Health Devices | Page 5), Internet of Medical Things (IoMT) (Trend 2 – Internet of Medical Things (IoMT) | Page 7), Artificial Intelligence (AI) (Slide 9), Telemedicine (Trend 4 – Telemedicine | Page 11),...
-------------------------------------------------------

Q: What are the future scopes of biomedical engineering?
A: The future scopes of biomedical engineering include smart hospitals, AI-driven diagnosis, personalized medicine, and fully automated healthcare systems (Future Scope | Page 18)....
-------------------------------------------------------


### Section 6: Expert-Guided Retrieval

#### The killer feature no one talks about.

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```text
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [37]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.

In [38]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [39]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [40]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "gpt-4o"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of the modern llm finetuning?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

In [ ]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Details of the syllabus of modern llm finetuning",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

### Section 7: Chat API — Zero LLM Setup

#### When to use this:

* You don't want to manage OpenAI API calls yourself
* You want a quick Q&A interface over your document
* You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [42]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here are the **key findings** from *Latest Trends in Biomedical Instrumentation*:

---

## 🔑 Core Trends Identified

| # | Trend | Highlights |
|---|-------|------------|
| 1 | **Wearable Health Devices** | Smartwatches & fitness bands (e.g., Apple Watch, Fitbit) for continuous heart rate, SpO₂, and sleep monitoring — enabling preventive care |
| 2 | **Internet of Medical Things (IoMT)** | Connected devices enabling Smart ICUs, remote patient monitoring, and home healthcare via real-time data sharing |
| 3 | **Artificial Intelligence (AI)** | Pattern recognition for cancer detection, ECG arrhythmia analysis, and radiology — reducing human error and speeding diagnosis |
| 4 | **Telemedicine** | Remote consultations via video conferencing and digital prescriptions, particularly impactful for rural healthcare access |
| 5 | **Point-of-Care (POC) Devices** | Portable tools (glucose monitors, portable ECG) for immediate, bedside diagnostics |
| 6 | **Implantable Devices**

In [ ]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What were the main revenue sources last year?",
    "How does that compare to the year before?",
    "What factors drove that change?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)

### Section 8: Self-Hosted Open Source Option

#### Use this when:

* You don't want to send documents to any cloud
* You need full data privacy / on-prem deployment
* You want to inspect or customize the tree-building logic

The open-source repo at `https://github.com/VectifyAI/PageIndex` lets you run the entire pipeline locally using your own OpenAI key.

#### What the CLI does:

1. Reads your PDF
2. Detects existing Table of Contents (if any)
3. Uses GPT-4o to build the hierarchical tree
4. Saves a `document_name_pageindex.json` alongside your PDF


In [ ]:
# ── Clone the open-source repo ───────────────────────────────────────────────
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

In [ ]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

import os
openai_key = os.getenv("OPENAI_API_KEY", "your_key_here")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print("✅ .env created for self-hosted mode")

In [ ]:
# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)

PDF_PATH = "/path/to/your/document.pdf"   # ← change this

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

In [ ]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

import json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

In [ ]:
# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the executive summary section."
answer = vectorless_rag(query, local_tree)

### Section 9: Vector RAG vs PageIndex — Side-by-Side

#### Architecture Comparison

| Aspect                | Traditional Vector RAG      | PageIndex (Vectorless RAG) |
| --------------------- | --------------------------- | -------------------------- |
| Document prep         | Chunk into fixed pieces     | Build hierarchical tree    |
| Indexing              | Embed each chunk            | LLM reads structure        |
| Storage               | Vector database             | JSON file                  |
| Query processing      | Embed query → ANN search    | LLM reasons over tree      |
| What's retrieved      | Flat anonymous chunks       | Named sections + page refs |
| Explainability        | ❌ Opaque similarity score   | ✅ Traceable reasoning      |
| Domain expertise      | ❌ Needs embedding fine-tune | ✅ Add rules to prompt      |
| Infrastructure        | Pinecone / FAISS / ChromaDB | No vector DB needed        |
| Best for              | Short, diverse documents    | Long, structured documents |
| FinanceBench accuracy | ~80%                        | **98.7%**                  |


#### When to use which

##### Use Vector RAG when:

* Documents are short and varied (FAQs, product descriptions)
* Semantic paraphrase matching is important
* You need sub-second retrieval on millions of documents

##### Use PageIndex when:

* Documents are long and professionally structured (reports, manuals, legal docs)
* You need traceable, cited answers
* Domain expertise should guide retrieval
* You want to avoid vector DB infrastructure


In [ ]:
# ── Quick comparison demo ────────────────────────────────────────────────────
# Show how the same query retrieves differently

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
""")

### Section 10: Cleanup

Delete documents from the PageIndex cloud when you're done
to keep your storage clean.

In [ ]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the tree index.
# Comment this out if you want to reuse the doc_id later.

# pi_client.delete_document(doc_id)
# print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")